In [1]:
import pandas as pd
import re 

pd.set_option('display.max_columns', None)

In [8]:
cdph = pd.read_csv('source/CDPH_Complaints_8722507853846220310.csv', low_memory=False)
cdph.columns = [x.lower().strip() for x in cdph.columns]
cdph = cdph.rename(columns={'dw_parcel': 'parcel'})

cdph[~cdph.odor_strength.isna()].sample(5)

,objectid,id,complaint_number,submit_date,submit_time,complaint_type,complaint_input,complaint_inspector,complaint_status,complaint_outcome,food_complaint,farm_animal,insect_vermin,odor_type,odor_strength,problem_location_name,problem_address,problem_street_direction,problem_street_name,problem_street_type,problem_unit_number,problem_city,problem_zip_code,problem_date_time,mac_complaint_id,permanent_parcel_number,census_tract,ward_number,submit_datetime,dw_ward,dw_ward_2014,dw_ward_2026,dw_neighborhood,dw_census_tract,parcel
13639,13640,8256,202108256,4/13/2021 12:00:00 AM,11:43AM,Odors,Sharia Clark,Jarett Gardner,Resolved,No Code Violation Observed,NaN,NaN,NaN,Animal Waste,10.0,NaN,2411,NaN,Garden,Ave,NaN,Cleveland,44109.0,04/13/2021-11:42am,NaN,NaN,1056,12.0,4/13/2021 3:43:00 PM,14,12,14,Brooklyn Centre,3.903511e+10,00824042
34200,34201,38416,2026038416,3/10/2026 12:00:00 AM,4:13PM,Animal Nuisance,Website Intake,Dontez Anderson,Active,Violation Notice Issued,NaN,NaN,NaN,Animal Waste,10.0,The Archer Apartments,1215,W,10th,St,NaN,Cleveland,44113.0,03/10/2026 4:11 PM,NaN,10113001,1071,7.0,3/10/2026 8:13:00 PM,7,3,7,Downtown,3.903511e+10,10113001
25912,25913,32327,2025032327,4/9/2025 12:00:00 AM,1:59PM,Open Burning,Website Intake,NaN,Resolved,NaN,NaN,NaN,NaN,Exhaust,10.0,NaN,3949,E,75th,St,NaN,Cleveland,44105.0,NaN,NaN,NaN,NaN,12.0,4/9/2025 5:59:00 PM,2,12,2,Broadway-Slavic Village,3.903512e+10,13316017
29875,29876,23643,2023023643,11/19/2023 12:00:00 AM,10:43AM,Open Burning,Website Intake,NaN,Resolved,NaN,NaN,NaN,NaN,Chemical,7.0,NaN,3692,W,129,St,NaN,Cleveland,44111.0,NaN,NaN,NaN,NaN,NaN,11/19/2023 3:43:00 PM,13,16,13,Jefferson,3.903512e+10,01828039
16316,16317,11867,2021011867,9/20/2021 12:00:00 AM,12:10PM,Odors,Sharia Clark,Luke Stover,Resolved,No Code Violation Observed,NaN,NaN,NaN,Sewage,10.0,NaN,3200,NaN,Tampa,Ave,NaN,Cleveland,44109.0,09/15/2021/09:30am,NaN,NaN,1068,13.0,9/20/2021 4:10:00 PM,4,13,4,Old Brooklyn,3.903511e+10,01103031


In [2]:
PAR_COLS = [
    'parcelpin', 
    'par_addr_all',
    'parcel_owner', 
    'std_deeded_owner', 
    'grantor', 
    'grantee',
    'last_transfer_date',
    'tax_assessed_total',
    'activerentalregistrationflag',
    'activecertificateapprovingrentaloccupancyflag',
    'leadsafecertificateactiveflag',
    'transfers_in_5y',
    'myplacelink',
    'survey2022_grade',
    'survey2022_photolink',
    'taxdelinquencyamount',
    'max_age',
    'min_age',
    'min_com_age',
    'max_com_age',
]

In [3]:
par = pd.read_csv('source/Property_Insights_Dataset.csv', low_memory=False)
par.columns = [x.lower().strip() for x in par.columns]
par = par[~par.par_addr_all.isna()]
par = par[PAR_COLS]
par = par.rename(columns={'parcelpin': 'parcel'})
par['owner_clean'] = par['std_deeded_owner'].apply(
    lambda x: re.sub(
        r"\s\s+",
        " ",
        re.sub(
            r"[^A-Za-z0-9]", 
            " ", 
            x
        )
    )
    if not pd.isna(x) else None
)
par = par.copy()

Civil Tickets

In [4]:
ct = pd.read_csv('source/Civil_Tickets.csv', low_memory=False)
ct.columns = [x.lower().strip() for x in ct.columns]
ct = ct.rename(columns={'dw_parcel': 'parcel'})
ct = ct[~ct.parcel.isna()].copy()

ct[[
    'ticket_id',
    'file_date',
    'ticket_status',
    'ticket_status_date',
    'issue_date',
    'additional_citation_details',
    'parcel',
]].to_csv('transformed/civil_tickets.csv', index=False)

ct_agg = ct.groupby('parcel').size().reset_index().rename(columns={0: 'civil_tickets'})
par = par.merge(ct_agg, on='parcel', how='left')

Health Complaints

In [5]:
cdph = pd.read_csv('source/CDPH_Complaints_8722507853846220310.csv', low_memory=False)
cdph.columns = [x.lower().strip() for x in cdph.columns]
cdph = cdph.rename(columns={'dw_parcel': 'parcel'})

cdph[[
    'id',
    'complaint_number',
    'submit_date',
    'complaint_type',
    'complaint_input',
    'complaint_inspector',
    'complaint_status',
    'complaint_outcome',
    'food_complaint',
    'farm_animal',
    'insect_vermin',
    'odor_type',
    'odor_strength',
    'problem_location_name',
    'parcel',
]].to_csv('transformed/health_complaints.csv', index=False)

cdph_agg = cdph.groupby('parcel').size().reset_index().rename(columns={0: 'health_complaints'})
par = par.merge(cdph_agg, on='parcel', how='left')

Violations

In [35]:
cmp_v = pd.read_csv('source/Complaint_Violation_Notices_8554063467594443398.csv', low_memory=False)
cmp_v.columns = [x.lower().strip() for x in cmp_v.columns]
cmp_v = cmp_v[[
    'violation_number',
    'record_id',
]].rename(columns={
    'record_id': 'complaint_id',
    'violation_number': 'record_id',
})
cmp_v = cmp_v.drop_duplicates('record_id')

v = pd.read_csv('source/Violation_Status_History_7194990999622183182.csv', low_memory=False)
v.columns = [x.lower().strip() for x in v.columns]

v = v[~v.task_status.isna()]
v['task_status'] = v['task_status'].apply(lambda x: x.strip())
v = v[v.task_status == 'VN Created/Mailed']

v = (v
    [[
        'record_id', 
        'dw_parcel',
        'type_of_violation',
        'occupancy_or_use',
        'issue_date',
        'accela_citizen_access_url',
    ]]
    .sort_values(['record_id', 'dw_parcel'])
    .drop_duplicates()
    .rename(columns={'dw_parcel': 'parcel'})
)
v = v.merge(cmp_v, on='record_id', how='left')
v.to_csv('transformed/code_violations.csv', index=False)

v_agg = v.groupby('parcel').size().reset_index().rename(columns={0: 'code_violations'})
par = par.merge(v_agg, on='parcel', how='left')

311 requests

In [36]:
req = pd.read_csv('source/Data_311_-3672549084735747031.csv', low_memory=False)
req.columns = ['_'.join(x.lower().strip().split()) for x in req.columns]
req = req.rename(columns={'parcelpin': 'parcel'})

req = req[~req.parcel.isna()]
req = req[req.agency_responsible == 'Building & Housing']

req[[
    'reference_number',
    'service_name',
    'status_description',
    'requested_datetime',
    'closed_date',
    'source',
    'target_date',
    'parcel',
]].to_csv('transformed/complaints_311.csv', index=False)

req_agg = (req
    .groupby('parcel')
    .size()
    .reset_index()
    .rename(columns={0: 'complaints_311'})
)
par = par.merge(req_agg, on='parcel', how='left')

# types = {
#     (
#         'General Exterior Maintenance/Fence/Outbuildings',
#         'Collapsing Structure',
#     ): 'exterior_maintenance_complaints',
#     (
#         'Vacant Property', 
#         'Vacant Property Open to Entry',
#         'Vacant and Distressed Property',
#     ): 'vacant_property_complaints',
#     ('Rental Inspection Complaint',): 'rental_inspection_complaints',
#     ('No Permit',): 'no_permit_complaints',
#     ('No Heat',): 'no_heat_complaints',
#     ('Grass/Weeds/Bushes (Occupied)',): 'high_grass_complaints',
#     ('Large Set Out or Eviction',): 'large_set_out_or_eviction_complaints',
#     ('Illegal Use of Property',): 'illegal_use_of_property_complaints',
# }
# # NOTE: This doesn't cover everything yet.

# serieses = []
# for type_tup, col_name in types.items():
#     serieses.append(req
#         [req.service_name.isin(type_tup)]
#         .groupby('parcel')
#         .size()
#         .reset_index()
#         .rename(columns={0: col_name})
#     )

# while serieses:
#     par = par.merge(serieses.pop(), on='parcel', how='left')

Clean parcels

In [37]:
metrics = [
    'code_violations', 
    'health_complaints', 
    'civil_tickets',
    'complaints_311',
]
# metrics.extend([x for x in types.values()])

par[metrics] = par[metrics].fillna(value=0)

In [38]:
par.to_csv('transformed/parcels.csv', index=False)

In [39]:
# su = par.groupby('owner_clean').sum(metrics)
# parcels = par.groupby('owner_clean').size().reset_index().rename(columns={0: 'parcels'})
# su['metrics_total'] = su.sum(axis=1)
# su = su.reset_index()
# su = su.merge(parcels, on='owner_clean', how='left')
# su['metrics_total_over_parcels'] = su.metrics_total / su.parcels

# # su.sort_values('metrics_total_over_parcels').tail(100)

# su.to_csv('transformed/owner_summary.csv', index=False)